# Reorder AstroFlame data and convert known units to CGS

This notebook reads `astroflame_unformatted_2.csv` and creates one CSV with one row per observation:

- Key measurement fields (thickness, width, oxygen percentages, and fire-spread rate) appear first.
- Other fields that vary within an experiment follow, with experiment-constant details such as material and geometry at the end.

No rows or values are removed. Millimeters are converted to centimeters and mm/s values to cm/s. Values already in cm or cm/s are unchanged. Oxygen percentages remain percentages because they are dimensionless. This source has no separate burn-rate or mass column, so neither is added or inferred.

In [12]:
from pathlib import Path
import pandas as pd

In [13]:
input_name = "astroflame_unformatted_2.csv"
candidate_dirs = [Path.cwd(), Path.cwd() / "elig-data"]
data_dir = next((path for path in candidate_dirs if (path / input_name).is_file()), None)
if data_dir is None:
    raise FileNotFoundError(f"Could not find {input_name} in the current folder or its elig-data subfolder.")

unformatted = data_dir / input_name
formatted = data_dir / "astroflame_formatted_cgs.csv"

In [14]:
source = pd.read_csv(unformatted)
required_columns = {
    "dataset",
    "experiment_id",
    "observation_id",
    "thickness_mm",
    "fsr_mm_s",
}
missing_columns = required_columns.difference(source.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")
if source.empty:
    raise ValueError("The input CSV contains no observations.")

df = source.copy()
for column in ["thickness_mm", "fsr_mm_s"]:
    df[column] = pd.to_numeric(df[column], errors="raise")

df["thickness_cm"] = df.pop("thickness_mm") / 10
df["fsr_cm_s"] = df.pop("fsr_mm_s") / 10

group_keys = ["dataset", "experiment_id"]
if df[group_keys].isna().any().any() or df["observation_id"].isna().any():
    raise ValueError("Dataset, experiment, and observation identifiers must not be blank.")

key_columns = [
    column
    for column in [
        "thickness_cm",
        "sample_width_cm",
        "sample_length_cm",
        "oxygen_percent",
        "oxygen_initial_pct",
        "oxygen_final_pct",
        "fsr_cm_s",
    ]
    if column in df.columns
]
id_columns = ["dataset", "experiment_id", "observation_id"]
remaining_columns = [
    column for column in df.columns if column not in key_columns + id_columns
]
varies_within_experiment = (
    df.groupby(group_keys, dropna=False)[remaining_columns]
    .nunique(dropna=False)
    .gt(1)
    .any()
)
changing_columns = [
    column for column in remaining_columns if varies_within_experiment[column]
]
constant_columns = [
    column for column in remaining_columns if not varies_within_experiment[column]
]
df = df[key_columns + id_columns + changing_columns + constant_columns]

if len(df) != len(source):
    raise AssertionError("Observation rows were lost while reordering columns.")
if df["observation_id"].tolist() != source["observation_id"].tolist():
    raise AssertionError("Observation order or identifiers changed unexpectedly.")

df.to_csv(formatted, index=False)
print(f"Read and retained {len(df)} observations.")
print(f"Wrote one CSV: {formatted.name}")
print(f"Key columns first: {key_columns}")
print(f"Other changing columns next: {changing_columns}")
print(f"Experiment-constant columns last: {constant_columns}")
print("Converted thickness_mm to thickness_cm and fsr_mm_s to fsr_cm_s; oxygen percentages remain unchanged.")

df.head()

Read and retained 44 observations.
Wrote one CSV: astroflame_formatted_cgs.csv
Key columns first: ['thickness_cm', 'sample_width_cm', 'sample_length_cm', 'oxygen_percent', 'oxygen_initial_pct', 'oxygen_final_pct', 'fsr_cm_s']
Other changing columns next: ['flow_direction', 'airflow_cm_s', 'measurement_method', 'source_location', 'notes', 'original_segment_number']
Experiment-constant columns last: ['material', 'geometry', 'oxygen_basis', 'burning_sides']
Converted thickness_mm to thickness_cm and fsr_mm_s to fsr_cm_s; oxygen percentages remain unchanged.


,thickness_cm,sample_width_cm,sample_length_cm,oxygen_percent,oxygen_initial_pct,oxygen_final_pct,fsr_cm_s,dataset,experiment_id,observation_id,flow_direction,airflow_cm_s,measurement_method,source_location,notes,original_segment_number,material,geometry,oxygen_basis,burning_sides
0,0.1,2.2,NaN,NaN,22.2,21.9,0.0071,BASS-II PSI-25,M1,BASSII_M1_seg1,opposed,9,reported_table,NASA/TM-20210011385 Table 5.1 p.57,Test-level oxygen is not segment oxygen,1.0,PMMA,flat slab,segment oxygen unknown; test start/end recorde...,1.0
1,0.3,2.2,NaN,NaN,22.2,21.2,0.0052,BASS-II PSI-25,M2,BASSII_M2_seg1,opposed,20,reported_table,NASA/TM-20210011385 Table 5.1 p.57,Test-level oxygen is not segment oxygen,1.0,PMMA,flat slab,segment oxygen unknown; test start/end recorde...,1.0
2,0.3,2.2,NaN,NaN,22.2,21.2,0.0033,BASS-II PSI-25,M2,BASSII_M2_seg2,opposed,15,reported_table,NASA/TM-20210011385 Table 5.1 p.57,Test-level oxygen is not segment oxygen,2.0,PMMA,flat slab,segment oxygen unknown; test start/end recorde...,1.0
3,0.5,2.2,NaN,NaN,20.8,19.7,0.0029,BASS-II PSI-25,M3,BASSII_M3_seg1,opposed,10,reported_table,NASA/TM-20210011385 Table 5.1 p.57,Test-level oxygen is not segment oxygen,1.0,PMMA,flat slab,segment oxygen unknown; test start/end recorde...,1.0
4,0.5,2.2,NaN,NaN,20.8,19.7,0.0024,BASS-II PSI-25,M3,BASSII_M3_seg2,opposed,20,reported_table,NASA/TM-20210011385 Table 5.1 p.57,Test-level oxygen is not segment oxygen,2.0,PMMA,flat slab,segment oxygen unknown; test start/end recorde...,1.0
